# Dixon Noise and variance demo

Demonstrates `estimate_dixon_noise` and `local_noise_map` from `dissector.diffusion`
on a real water / fat stack pair from myosegmenTUM. This was a first attempt at noise harvesting

In [ ]:
import os
import glob
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

from dissector.diffusion import estimate_dixon_noise, local_noise_map

In [ ]:
# pick the first available water/fat stack pair
DATA_ROOT = 'myosegmenTUM'

water_files = sorted(glob.glob(os.path.join(DATA_ROOT, '*', 'ImageData', '*_WATER', '*_WATER_stack1.nii')))
fat_files   = sorted(glob.glob(os.path.join(DATA_ROOT, '*', 'ImageData', '*_FAT',   '*_FAT_stack1.nii')))

print(f'Water stacks found: {len(water_files)}')
print(f'Fat stacks found  : {len(fat_files)}')

water_path = water_files[0]
# match the same subject for fat
subject    = os.path.basename(water_path).split('_WATER')[0]
fat_path   = next(f for f in fat_files if os.path.basename(f).startswith(subject))

print(f'\nUsing water: {water_path}')
print(f'Using fat  : {fat_path}')

In [ ]:
water_arr = sitk.GetArrayFromImage(sitk.ReadImage(water_path)).astype(float)
fat_arr   = sitk.GetArrayFromImage(sitk.ReadImage(fat_path)).astype(float)

print(f'Water shape: {water_arr.shape}  min={water_arr.min():.1f}  max={water_arr.max():.1f}')
print(f'Fat shape  : {fat_arr.shape}  min={fat_arr.min():.1f}  max={fat_arr.max():.1f}')

In [ ]:
# show a mid-volume slice from each channel
mid = water_arr.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(water_arr[mid], cmap='gray', origin='lower')
axes[0].set_title(f'Water — slice {mid}')
axes[0].axis('off')

axes[1].imshow(fat_arr[mid], cmap='gray', origin='lower')
axes[1].set_title(f'Fat — slice {mid}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## `estimate_dixon_noise`

Finds background voxels (both channels below the 10th percentile) and computes
noise σ and SNR for each channel. `noise_ratio` ≈ 1 means the two channels
have matched noise floors — a sign of a clean Dixon reconstruction.

In [ ]:
stats = estimate_dixon_noise(water_arr, fat_arr)

print('─── Dixon noise estimate ───')
for k, v in stats.items():
    print(f'  {k:<25s}: {v:.4f}' if isinstance(v, float) else f'  {k:<25s}: {v}')

In [ ]:
# visualise the background mask used for noise estimation
bg_pct   = 10.0
w_thresh = np.percentile(water_arr, bg_pct)
f_thresh = np.percentile(fat_arr,   bg_pct)
bg_mask  = (water_arr <= w_thresh) & (fat_arr <= f_thresh)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(water_arr[mid], cmap='gray', origin='lower')
axes[0].imshow(bg_mask[mid], cmap='Reds', alpha=0.4, origin='lower')
axes[0].set_title('Water + background mask (red)')
axes[0].axis('off')

axes[1].imshow(fat_arr[mid], cmap='gray', origin='lower')
axes[1].imshow(bg_mask[mid], cmap='Reds', alpha=0.4, origin='lower')
axes[1].set_title('Fat + background mask (red)')
axes[1].axis('off')

plt.suptitle(f'Background voxels: {bg_mask.sum()} ({100*bg_mask.mean():.1f}%)', fontsize=11)
plt.tight_layout()
plt.show()

## `local_variance_map`

Computes a rolling local standard deviation over a cubic neighbourhood.
High values might indicate spatially varying noise or reconstruction artefacts.Or maybe tissue broders.
Run on a single slice here to keep it fast.

In [ ]:
# run on a 3-slice slab around mid to keep it quick
slab       = water_arr[mid-1 : mid+2]
noise_map  = local_noise_map(slab, kernel_size=5)
noise_slice = noise_map[1]  # centre slice of the slab

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(water_arr[mid], cmap='gray', origin='lower')
axes[0].set_title('Water image')
axes[0].axis('off')

im = axes[1].imshow(noise_slice, cmap='hot', origin='lower')
axes[1].set_title('Local noise map (water, kernel=5)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# overlay: noise map on top of water image
water_norm = (water_arr[mid] - water_arr[mid].min()) / (water_arr[mid].max() - water_arr[mid].min() + 1e-8)
noise_norm = (noise_slice - noise_slice.min()) / (noise_slice.max() - noise_slice.min() + 1e-8)
axes[2].imshow(water_norm, cmap='gray', origin='lower')
axes[2].imshow(noise_norm, cmap='hot', alpha=0.5, origin='lower')
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.suptitle(f'Local noise  mean={noise_slice.mean():.2f}  max={noise_slice.max():.2f}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# compare water vs fat noise maps side by side
fat_slab      = fat_arr[mid-1 : mid+2]
fat_noise_map = local_noise_map(fat_slab, kernel_size=5)
fat_noise_slice = fat_noise_map[1]

vmax = max(noise_slice.max(), fat_noise_slice.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(noise_slice,     cmap='hot', origin='lower', vmin=0, vmax=vmax)
axes[0].set_title('Water local variance')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(fat_noise_slice, cmap='hot', origin='lower', vmin=0, vmax=vmax)
axes[1].set_title('Fat local variance')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('Shared colour scale — differences may indicate channel-specific artefacts', fontsize=10)
plt.tight_layout()
plt.show()